# Image Recognition: Classifying Images Based on Nearby Feature Representations
### Practical Demonstration of K-Nearest Neighbors (KNN) — Google Colab

**Course:** Artificial Intelligence  
**Course Teacher:** Ahsan Ullah, Assistant Professor, Dept. of CSE, World University of Bangladesh  
**Primary reference:** Lecture notes *K-Nearest Neighbors (KNN)* (`KNN.md`)

**Presented by:**
- Md Azadur Rahman (ID: 4478)
- Rakib Hossain (ID: 4056)
- Aduri Akter (ID: 4032)

---

This notebook follows the **same steps as the teacher's KNN lecture**, applied to images:

> **Image → Feature Representation → Distance Calculation → Nearest Neighbors → Classification**

| Notebook section | Lecture section (KNN.md) |
|---|---|
| 2. Slide examples reproduced | Worked Example, Multi-feature Example, Feature Scaling, KNN Regression |
| 3. Images as feature vectors | Applications → *Image recognition* |
| 4. Training data and test data | Training dataset / test data |
| 5–8. Steps 1–5 of KNN on a real image | *KNN Classification Process* |
| 9. Python implementation | Python code sections |
| 10. Accuracy | Applying KNN to a classification problem |
| 11. Choosing K | *Choosing the Value of K* |
| 12. Feature scaling | *Feature Scaling in KNN* |
| 14. Summary | *Advantages and Limitations* |

Runs on a normal CPU. No GPU, no download, and no local file paths are needed.

---
## 1. Environment Setup
We only need **NumPy** (numbers), **Matplotlib** (plots) and **Scikit-learn** (dataset and KNN). All three are pre-installed in Google Colab.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

np.set_printoptions(suppress=True)
plt.rcParams["figure.dpi"] = 100

print("NumPy       :", np.__version__)
print("Scikit-learn:", sklearn.__version__)

ModuleNotFoundError: No module named 'matplotlib'

---
## 2. Reproducing the Slide Examples
Before using real images, we check every hand calculation from the slides with Python, exactly like the teacher's worked examples.

### 2.1 Worked Example 1 — One image feature (mirrors the *study hours* example)
Feature = **number of dark pixels** in a small digit image. With only one feature, the distance is the absolute difference:

$$d = |x_{new} - x_{training}|$$

| Image | Dark pixels | Class |
|---|---|---|
| A | 5 | Digit 1 |
| B | 6 | Digit 1 |
| C | 12 | Digit 0 |
| D | 13 | Digit 0 |
| E | 14 | Digit 0 |

A new image has **8** dark pixels. Predict its class with **K = 3**.

In [ ]:
names = ["A", "B", "C", "D", "E"]
dark_pixels = np.array([5, 6, 12, 13, 14])
labels_1 = np.array([1, 1, 0, 0, 0])
x_new = 8
K = 3

dist_1 = np.abs(x_new - dark_pixels)
print("Step 1: Calculate the distances  d = |8 - x|")
for n, x, d, c in zip(names, dark_pixels, dist_1, labels_1):
    print(f"  Image {n}: |8 - {x:2d}| = {d}   -> Digit {c}")

nearest = np.argsort(dist_1, kind="stable")[:K]
print("\nStep 2: The 3 nearest images:", [names[i] for i in nearest],
      "with classes", [f"Digit {labels_1[i]}" for i in nearest])

votes = np.bincount(labels_1[nearest], minlength=2)
print(f"\nStep 3: Majority voting -> Digit 0: {votes[0]} vote(s), Digit 1: {votes[1]} vote(s)")
print("Predicted class = Digit", votes.argmax())

# Same answer with scikit-learn
model_1 = KNeighborsClassifier(n_neighbors=3).fit(dark_pixels.reshape(-1, 1), labels_1)
print("Scikit-learn prediction = Digit", model_1.predict([[x_new]])[0])

### 2.2 Worked Example 2 — Multiple image features (Euclidean distance)
Each image is described by three features (scale 0–10): $x_1$ = **Center Ink**, $x_2$ = **Loop Score**, $x_3$ = **Vertical Stroke**.

$$d(A, B) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

New image: $X_{new} = (3, 6, 4)$. We also try **K = 1, 3 and 5**, as the teacher suggests.

In [ ]:
X2 = np.array([[2, 8, 3],   # A -> Digit 0
               [3, 7, 4],   # B -> Digit 0
               [8, 2, 9],   # C -> Digit 1
               [7, 3, 8],   # D -> Digit 1
               [9, 1, 9]])  # E -> Digit 1
y2 = np.array([0, 0, 1, 1, 1])
x_new_2 = np.array([3, 6, 4])

dist_2 = np.sqrt(((X2 - x_new_2) ** 2).sum(axis=1))
print("Euclidean distances from X_new = (3, 6, 4):")
for n, v, d, c in zip(names, X2, dist_2, y2):
    sq = " + ".join(f"({a}-{b})^2" for a, b in zip(x_new_2, v))
    print(f"  Image {n} {tuple(v.tolist())}: sqrt({sq}) = {d:.2f}  -> Digit {c}")

print()
for k in (1, 3, 5):
    nn = np.argsort(dist_2, kind="stable")[:k]
    pred = KNeighborsClassifier(n_neighbors=k).fit(X2, y2).predict([x_new_2])[0]
    print(f"K = {k}: neighbors {[names[i] for i in nn]} -> Predicted class = Digit {pred}")

**Observation:** K = 1 and K = 3 predict **Digit 0**. With **K = 5**, all five training images vote (2 × Digit 0 vs. 3 × Digit 1) and the answer flips to Digit 1. Too large a K brings in far-away images; this is **underfitting** (see *Choosing the Value of K*).

### 2.3 Worked Example 3 — Raw pixels as the feature representation
Tiny 3×3 black-and-white images (1 = ink, 0 = blank) are flattened row-by-row into 9-number vectors.

In [ ]:
pix_train = np.array([
    [0,1,0, 0,1,0, 0,1,0],   # "1" (a)
    [1,1,0, 0,1,0, 0,1,0],   # "1" (b) with a small top flag
    [1,1,1, 1,0,1, 1,1,1],   # "0" (a)
    [1,1,1, 1,0,1, 1,1,0],   # "0" (b) with a missing corner
])
pix_labels = np.array([1, 1, 0, 0])
pix_names = ['"1"(a)', '"1"(b)', '"0"(a)', '"0"(b)']
pix_query = np.array([0,1,0, 0,1,0, 1,1,0])   # a noisy "1"

fig, axes = plt.subplots(1, 5, figsize=(9, 2.2))
for ax, img, title in zip(axes, list(pix_train) + [pix_query], pix_names + ["Query ?"]):
    ax.imshow(img.reshape(3, 3), cmap="Greys", vmin=0, vmax=1)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

dist_3 = np.sqrt(((pix_train - pix_query) ** 2).sum(axis=1))
for n, d, s in zip(pix_names, dist_3, ((pix_train - pix_query) ** 2).sum(axis=1)):
    print(f"d(query, {n}) = sqrt({s}) = {d:.2f}")
print("K = 3 prediction = Digit", KNeighborsClassifier(n_neighbors=3).fit(pix_train, pix_labels).predict([pix_query])[0])

### 2.4 Worked Example 4 — Why feature scaling matters (mirrors *age vs. income*)
Two image features with very different ranges:
- **Total ink** (range 0–1024)
- **Symmetry** (range 0–1)

| Image | Total ink | Symmetry | Class |
|---|---|---|---|
| P | 520 | 0.90 | Digit 0 (round, symmetric) |
| Q | 525 | 0.20 | Digit 7 (not symmetric) |

New image: (530, 0.85), which is very symmetric like a 0. We use K = 1 (the single nearest image).

In [ ]:
S = np.array([[520, 0.90],    # P -> Digit 0
              [525, 0.20]])   # Q -> Digit 7
s_labels = ["Digit 0", "Digit 7"]
s_new = np.array([530, 0.85])

d_raw = np.sqrt(((S - s_new) ** 2).sum(axis=1))
print("Without scaling:  d(P) = %.4f, d(Q) = %.4f  -> nearest = %s" % (d_raw[0], d_raw[1], s_labels[d_raw.argmin()]))

feature_range = np.array([1024.0, 1.0])          # normalize each feature by its range
d_norm = np.sqrt((((S - s_new) / feature_range) ** 2).sum(axis=1))
print("With normalization: d(P) = %.4f, d(Q) = %.4f  -> nearest = %s" % (d_norm[0], d_norm[1], s_labels[d_norm.argmin()]))

Without scaling, the *total ink* feature dominates the distance and the symmetry feature is ignored, so the prediction is wrong. After normalization both features count fairly.

### 2.5 KNN Regression with images (mirrors the *house price* example)
KNN can also predict a **number**. Illustrative example: estimate a person's **age** from one face-image feature, a *texture score* (0–10). *(The numbers are made up for teaching only.)*

| Face image | Texture score | Age (years) |
|---|---|---|
| A | 1 | 20 |
| B | 2 | 25 |
| C | 4 | 35 |
| D | 6 | 45 |
| E | 8 | 60 |

New face: texture score = 3, K = 3.

In [ ]:
tex = np.array([1, 2, 4, 6, 8])
age = np.array([20, 25, 35, 45, 60])
tex_new = 3

d_reg = np.abs(tex_new - tex)
nn_reg = np.argsort(d_reg, kind="stable")[:3]
print("Distances:", dict(zip(names, d_reg.tolist())))
print("3 nearest faces:", [names[i] for i in nn_reg], "ages", age[nn_reg].tolist())
print("Prediction = (%s) / 3 = %.2f years" % (" + ".join(map(str, age[nn_reg])), age[nn_reg].mean()))

reg = KNeighborsRegressor(n_neighbors=3).fit(tex.reshape(-1, 1), age)
print("Scikit-learn KNeighborsRegressor:", round(reg.predict([[tex_new]])[0], 2), "years")

---
## 3. Real Images as Feature Representations
We use Scikit-learn's **handwritten digits** dataset:
- **1,797** grayscale images of the digits **0–9**
- each image is **8 × 8 = 64 pixels**
- each pixel value runs from **0 (white)** to **16 (dark ink)**

The **feature representation** of an image is its 64 pixel values written as one list (a 64-dimensional vector), in the same way the teacher's fruit is described by (weight, size).

In [ ]:
digits = load_digits()
images = digits.images        # (1797, 8, 8)  -> image grids
X = digits.data               # (1797, 64)    -> feature vectors
y = digits.target             # (1797,)       -> class labels 0..9

print("Number of images :", X.shape[0])
print("Image grid shape :", images.shape[1:])
print("Feature vector   :", X.shape[1], "numbers per image")

fig, axes = plt.subplots(1, 10, figsize=(12, 1.8))
for d, ax in enumerate(axes):
    ax.imshow(images[np.where(y == d)[0][0]], cmap="Greys")
    ax.set_title(f"Digit {d}"); ax.axis("off")
plt.suptitle("One example of each class", fontweight="bold"); plt.tight_layout(); plt.show()

### From image → pixel grid → feature vector
Below, the same image is shown three ways: as we see it, as the computer stores it (an 8 × 8 grid of numbers), and as the **64-number feature vector** that KNN uses.

In [ ]:
i = 0
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), gridspec_kw={"width_ratios": [1, 1, 3]})
axes[0].imshow(images[i], cmap="Greys"); axes[0].set_title(f"Image (Digit {y[i]})"); axes[0].axis("off")

axes[1].imshow(images[i], cmap="Greys", vmin=0, vmax=16)
for r in range(8):
    for c in range(8):
        v = int(images[i][r, c])
        axes[1].text(c, r, v, ha="center", va="center", fontsize=7, color="white" if v > 8 else "black")
axes[1].set_title("8 x 8 pixel values"); axes[1].axis("off")

axes[2].bar(range(64), X[i], color="steelblue")
axes[2].set_title("Feature vector: 64 numbers (pixels read row by row)")
axes[2].set_xlabel("Feature index (0..63)"); axes[2].set_ylabel("Pixel value (0..16)")
plt.tight_layout(); plt.show()

print("First 16 features:", X[i][:16].astype(int).tolist(), "...")

---
## 4. Training Data and Test Data
As in the lecture, KNN needs a **training dataset** (images with known labels). To check the predictions fairly, we keep **25%** of the images aside as **test data**: new images whose labels the model does not see.

In [ ]:
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    X, y, images, test_size=0.25, random_state=42, stratify=y)

print("Training images:", len(X_train))
print("Test images    :", len(X_test))

---
## 5. KNN Step by Step on a Real Image
We now follow the teacher's **KNN Classification Process** by hand, using NumPy only.

**Step 1 — Store the training data.** KNN simply keeps the training feature vectors and their labels (*"no explicit model training is required"*).  
**Step 2 — Choose a value of K.** We choose **K = 3**.

In [ ]:
stored_X, stored_y = X_train, y_train     # Step 1: store
K = 3                                      # Step 2: choose K

query_index = 0                            # a test image the model has never seen
query = X_test[query_index]

plt.figure(figsize=(2, 2)); plt.imshow(img_test[query_index], cmap="Greys"); plt.axis("off")
plt.title("New image ?"); plt.show()
print("Stored training vectors:", stored_X.shape, "| K =", K)

**Step 3 — Calculate the distance** between the new image and **every** training image with the Euclidean formula over all $n = 64$ features:

$$d(A, B) = \sqrt{\sum_{i=1}^{64} (a_i - b_i)^2}$$

In [ ]:
distances = np.sqrt(((stored_X - query) ** 2).sum(axis=1))   # one distance per training image
print("Number of distances calculated:", len(distances))

order = np.argsort(distances, kind="stable")
print("\nThe 5 smallest distances:")
print(f"{'Rank':<6}{'Training image':<16}{'Distance':<10}{'Class'}")
for rank, j in enumerate(order[:5], start=1):
    print(f"{rank:<6}{'#' + str(j):<16}{distances[j]:<10.2f}Digit {stored_y[j]}")

**Step 4 — Select the K nearest neighbors.** These are the K = 3 training images with the smallest distances.

In [ ]:
nearest_k = order[:K]
fig, axes = plt.subplots(1, K + 1, figsize=(8, 2.3))
axes[0].imshow(img_test[query_index], cmap="Purples"); axes[0].set_title("New image ?"); axes[0].axis("off")
for ax, j in zip(axes[1:], nearest_k):
    ax.imshow(img_train[j], cmap="Greys")
    ax.set_title(f"Digit {stored_y[j]}\nd = {distances[j]:.2f}"); ax.axis("off")
plt.suptitle("The K = 3 nearest neighbors", fontweight="bold"); plt.tight_layout(); plt.show()

**Step 5 — Majority voting.** The class with the most votes among the K neighbors is the predicted class.

In [ ]:
votes = np.bincount(stored_y[nearest_k], minlength=10)
for digit in np.nonzero(votes)[0]:
    print(f"Digit {digit}: {votes[digit]} vote(s)")
manual_prediction = votes.argmax()
print("\nPredicted class =", f"Digit {manual_prediction}")
print("True class      =", f"Digit {y_test[query_index]}")

---
## 6. The Same Thing with Scikit-learn
The teacher's lecture uses scikit-learn's KNN models. `KNeighborsClassifier` does Steps 1–5 for us. It should give exactly the same neighbors and prediction as our manual calculation.

In [ ]:
model = KNeighborsClassifier(n_neighbors=3)     # Step 2: K = 3 (Euclidean distance by default)
model.fit(X_train, y_train)                      # Step 1: store the training data

sk_dist, sk_idx = model.kneighbors([query])      # Steps 3-4
sk_pred = model.predict([query])[0]              # Step 5

print("Scikit-learn neighbors :", sk_idx[0].tolist(), "distances", np.round(sk_dist[0], 2).tolist())
print("Manual neighbors       :", nearest_k.tolist(), "distances", np.round(distances[nearest_k], 2).tolist())
print("Scikit-learn prediction: Digit", sk_pred, "| Manual prediction: Digit", manual_prediction)

---
## 7. Classifying All Test Images
Now we predict **every** test image and count how many predictions are correct.

$$\text{Accuracy} = \frac{\text{correct predictions}}{\text{number of test images}}$$

As the lecture reminds us, each answer is a prediction based on the training data, **not a guarantee**.

In [ ]:
y_pred = model.predict(X_test)
correct = (y_pred == y_test).sum()
print(f"Correct: {correct} / {len(y_test)}  ->  Accuracy = {accuracy_score(y_test, y_pred) * 100:.2f}%")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, j in zip(axes, range(8)):
    ax.imshow(img_test[j], cmap="Greys"); ax.axis("off")
    ax.set_title(f"Pred {y_pred[j]}\nTrue {y_test[j]}", color="green" if y_pred[j] == y_test[j] else "red")
plt.suptitle("Predictions on test images (K = 3)", fontweight="bold"); plt.tight_layout(); plt.show()

### Which digits get mixed up?
The table below (a *confusion matrix*) shows the true digit in each row and the predicted digit in each column. Numbers off the diagonal are mistakes. We then look at the misclassified images and their nearest neighbors to see **why** KNN made each mistake.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", colorbar=False, ax=ax)
ax.set_title("True digit (rows) vs. predicted digit (columns)"); plt.tight_layout(); plt.show()

wrong = np.where(y_pred != y_test)[0]
print("Misclassified test images:", len(wrong))
_, w_idx = model.kneighbors(X_test[wrong[:3]])
fig, axes = plt.subplots(min(3, len(wrong)), 4, figsize=(7, 5.5))
for r, j in enumerate(wrong[:3]):
    axes[r, 0].imshow(img_test[j], cmap="Reds"); axes[r, 0].set_title(f"True {y_test[j]}, Pred {y_pred[j]}")
    for c in range(3):
        axes[r, c + 1].imshow(img_train[w_idx[r, c]], cmap="Greys"); axes[r, c + 1].set_title(f"Neighbor: {y_train[w_idx[r, c]]}")
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("Mistakes and their 3 nearest neighbors", fontweight="bold"); plt.tight_layout(); plt.show()

---
## 8. Choosing the Value of K
From the lecture: a **small K** is sensitive to noise (**overfitting**), while a **large K** brings in far-away neighbors (**underfitting**). In practice, K is chosen with validation data or cross-validation. Here we simply compare test accuracy for several K values.

In [ ]:
k_values = [1, 3, 5, 7, 9, 15, 25, 51, 101]
acc = []
for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    acc.append(accuracy_score(y_test, m.predict(X_test)) * 100)
    print(f"K = {k:3d}  ->  accuracy = {acc[-1]:.2f}%")

plt.figure(figsize=(7, 3.5))
plt.plot(k_values, acc, "o-")
plt.xscale("log"); plt.xticks(k_values, k_values)
plt.xlabel("K (number of neighbors)"); plt.ylabel("Test accuracy (%)")
plt.title("Effect of K on accuracy"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**Reading the plot:** small K values (3–5) give the best accuracy on this dataset. As K becomes very large, neighbors from other digit classes join the vote and accuracy falls (**underfitting**).

---
## 9. Feature Scaling in KNN
The lecture explains that features with **different ranges** must be scaled (standardization or normalization), as in Worked Example 4.

In the digits dataset, **all 64 features are pixels on the same 0–16 scale**, so no feature can dominate because of its units. Let us check what happens when we still apply `StandardScaler`, written the same way as the teacher's `make_pipeline` code.

In [ ]:
plain = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)).fit(X_train, y_train)

print(f"Raw pixels (same 0-16 range) : {accuracy_score(y_test, plain.predict(X_test)) * 100:.2f}%")
print(f"With StandardScaler          : {accuracy_score(y_test, scaled.predict(X_test)) * 100:.2f}%")

**Honest conclusion:** here scaling does **not** help, because the pixels already share one scale. Standardizing gives nearly-always-blank border pixels the same weight as the important center pixels. Scaling is essential when the features have **different units or ranges** (Worked Example 4, or the teacher's age-vs-income example). It is not automatically better.

---
## 10. Optional: Test Your Own Digit Image
- With `UPLOAD_OWN_IMAGE = False` (default), the cell uses a built-in hand-drawn "0" so *Run all* never stops.
- In Google Colab, set `UPLOAD_OWN_IMAGE = True` and run the cell to upload a PNG/JPG photo of **one** digit (dark ink on light paper works best).

The image is converted to the dataset's format (black-and-white 32 × 32 bitmap → number of ink pixels in each 4 × 4 block → 8 × 8 grid with values 0–16), and then KNN finds its 3 nearest training images.

In [ ]:
from PIL import Image, ImageDraw

def to_8x8_feature_vector(pil_img):
    # Convert a digit photo to the dataset's format, the same way the dataset was made:
    # black/white 32x32 bitmap -> count ink pixels in each 4x4 block -> 8x8 grid of values 0..16.
    arr = np.array(pil_img.convert("L"), dtype=float)
    if np.median(np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])) > 127:
        arr = 255 - arr                                  # make ink bright, background dark
    ink = arr > 0.5 * arr.max()                          # black/white bitmap
    ys, xs = np.where(ink)
    ink = ink[ys.min():ys.max() + 1, xs.min():xs.max() + 1]   # crop around the digit
    side = max(ink.shape)                                # center it in a square
    square = np.zeros((side, side), dtype=bool)
    oy, ox = (side - ink.shape[0]) // 2, (side - ink.shape[1]) // 2
    square[oy:oy + ink.shape[0], ox:ox + ink.shape[1]] = ink
    b32 = np.array(Image.fromarray(np.uint8(square) * 255).resize((32, 32), Image.Resampling.NEAREST)) > 127
    return b32.reshape(8, 4, 8, 4).sum(axis=(1, 3)).astype(float)

UPLOAD_OWN_IMAGE = False

own = None
if UPLOAD_OWN_IMAGE:
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            own = Image.open(next(iter(uploaded)))
    except ImportError:
        print("Upload works only in Google Colab - using the built-in example.")
if own is None:                                         # built-in hand-drawn "0"
    own = Image.new("L", (64, 64), 248)
    ImageDraw.Draw(own).ellipse([16, 8, 48, 56], outline=16, width=8)

own_8x8 = to_8x8_feature_vector(own)
own_dist, own_idx = model.kneighbors([own_8x8.ravel()])
own_pred = model.predict([own_8x8.ravel()])[0]

fig, axes = plt.subplots(1, 5, figsize=(10, 2.3))
axes[0].imshow(own, cmap="gray"); axes[0].set_title("Your image")
axes[1].imshow(own_8x8, cmap="Greys"); axes[1].set_title(f"8x8 -> Pred {own_pred}")
for ax, j, d in zip(axes[2:], own_idx[0], own_dist[0]):
    ax.imshow(img_train[j], cmap="Greys"); ax.set_title(f"Digit {y_train[j]}\nd = {d:.1f}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("Predicted class = Digit", own_pred, "| neighbor classes:", y_train[own_idx[0]].tolist())

---
## 11. Summary — Linking Back to the Lecture

1. **Image → Feature Representation:** every image becomes a list of numbers (here 64 pixel values), just as a fruit is described by its weight and size.
2. **Distance Calculation:** the Euclidean distance $d(A,B)=\sqrt{\sum (a_i-b_i)^2}$ measures how different two images are.
3. **Nearest Neighbors → Classification:** the K closest training images vote, and the majority class is the prediction. KNN can also average neighbor values for **regression**.
4. **Choosing K:** small K overfits noise and large K underfits. K = 3–5 worked best here.
5. **Feature scaling:** essential when features have different ranges, but not automatically helpful when all features are same-scale pixels.

| Advantages (lecture) | Seen in this notebook |
|---|---|
| Simple and easy to understand | Steps 1–5 done by hand with a few lines of NumPy |
| No explicit model training | `fit()` only stores the training images |
| Classification **and** regression | Digit classes and the age example |
| Nonlinear decision boundaries | 10 digit classes separated without any formula |

| Limitations (lecture) | Seen in this notebook |
|---|---|
| Slow prediction for large datasets | Every prediction compares against all 1,347 training images |
| Sensitive to feature scaling | Worked Example 4 |
| Depends on K | Accuracy changes with K |
| Sensitive to noise / irrelevant features | The misclassified, unusually written digits |

**Optional future work (beyond the lecture):** for real photographs, raw pixels are not a good enough representation. Modern systems first extract better features (for example with neural networks) and can then still use nearest-neighbor search on those features.